<a href="https://colab.research.google.com/github/Ankith020/Student-Dropout-Prediction/blob/main/ResearchProjectCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.feature_selection import SelectKBest, mutual_info_classif

PALETTE = sns.color_palette('viridis', n_colors=5)

In [2]:
df = pd.read_excel("/content/Research Project dataset.xlsx")
print("=" * 60)
print("DATASET LOADED")
print(f"  Shape : {df.shape[0]} rows × {df.shape[1]} columns")
print("=" * 60)

DATASET LOADED
  Shape : 649 rows × 34 columns


# **EDA(Exploratory Data Analysis)**

In [3]:
print("\n[1-A] Column Data Types")
print(df.dtypes.to_string())

print("\n[1-B] First 5 Rows")
print(df.head().to_string())

print("\n[1-C] Statistical Summary — Numerical Columns")
print(df.describe().round(2).to_string())

print("\n[1-D] Categorical Columns — Value Counts")
cat_cols = df.select_dtypes(include=["object", "bool"]).columns.tolist()
for col in cat_cols:
    print(f"\n  {col}:")
    print(df[col].value_counts().to_string())

print("\n[1-E] Missing Values")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "  ✔ No missing values found.")


[1-A] Column Data Types
School                         object
Gender                         object
Age                             int64
Address                        object
Family_Size                    object
Parental_Status                object
Mother_Education                int64
Father_Education                int64
Mother_Job                     object
Father_Job                     object
Reason_for_Choosing_School     object
Guardian                       object
Travel_Time                     int64
Study_Time                      int64
Number_of_Failures              int64
School_Support                 object
Family_Support                 object
Extra_Paid_Class               object
Extra_Curricular_Activities    object
Attended_Nursery               object
Wants_Higher_Education         object
Internet_Access                object
In_Relationship                object
Family_Relationship             int64
Free_Time                       int64
Going_Out                

# **EDA Visualizations**

In [4]:
fig = plt.figure(figsize=(20, 18))
fig.suptitle("Student Performance Dataset — EDA Overview", fontsize=18, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# Target class distribution
ax0 = fig.add_subplot(gs[0, 0])
counts = df["Dropped_Out"].value_counts().sort_index()
ax0.bar(["Not Dropped Out", "Dropped Out"], counts.values,
        color=["#2ecc71", "#e74c3c"], edgecolor="white", linewidth=1.2)
ax0.set_title("Target: Dropped Out Distribution", fontweight="bold")
ax0.set_ylabel("Count")
for i, v in enumerate(counts.values):
    ax0.text(i, v + 5, str(v), ha="center", fontweight="bold")

# Age distribution
ax1 = fig.add_subplot(gs[0, 1])
ax1.hist(df["Age"], bins=10, color="#3498db", edgecolor="white", alpha=0.85)
ax1.set_title("Age Distribution", fontweight="bold")
ax1.set_xlabel("Age"); ax1.set_ylabel("Count")

# Grade_1 distribution (early academic performance — used instead of Final_Grade)
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(df["Grade_1"], bins=20, color="#9b59b6", edgecolor="white", alpha=0.85)
ax2.set_title("Grade 1 Distribution (Early Performance)", fontweight="bold")
ax2.set_xlabel("Grade 1 (0–20)"); ax2.set_ylabel("Count")

# Gender vs Dropout rate
ax3 = fig.add_subplot(gs[1, 0])
gender_drop = df.groupby("Gender")["Dropped_Out"].value_counts(normalize=True).unstack() * 100
gender_drop.plot(kind="bar", ax=ax3, color=["#2ecc71", "#e74c3c"], edgecolor="white", rot=0)
ax3.set_title("Dropout Rate by Gender (%)", fontweight="bold")
ax3.set_ylabel("Percentage")
ax3.legend(["Not Dropped Out", "Dropped Out"])

# Study time vs Grade_1
ax4 = fig.add_subplot(gs[1, 1])
study_labels = {1: "< 2h", 2: "2–5h", 3: "5–10h", 4: "> 10h"}
df_plot = df.copy()
df_plot["Study_Time_Label"] = df["Study_Time"].map(study_labels)
sns.boxplot(data=df_plot, x="Study_Time_Label", y="Grade_1",
            order=["< 2h", "2–5h", "5–10h", "> 10h"], palette=PALETTE, ax=ax4)
ax4.set_title("Study Time vs Grade 1", fontweight="bold")
ax4.set_xlabel("Weekly Study Time"); ax4.set_ylabel("Grade 1")

# Number of Failures vs Grade_1
ax5 = fig.add_subplot(gs[1, 2])
sns.boxplot(data=df, x="Number_of_Failures", y="Grade_1", palette=PALETTE, ax=ax5)
ax5.set_title("Past Failures vs Grade 1", fontweight="bold")
ax5.set_xlabel("Number of Past Failures"); ax5.set_ylabel("Grade 1")

# Correlation heatmap — numerical features only
ax6 = fig.add_subplot(gs[2, :])
num_cols = df.select_dtypes(include=np.number).columns.tolist()
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, ax=ax6, annot_kws={"size": 7})
ax6.set_title("Correlation Matrix — Numerical Features", fontweight="bold")

plt.savefig("eda_overview.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → Saved: eda_overview.png")

# Deep-dive plots
fig2, axes = plt.subplots(2, 3, figsize=(18, 10))
fig2.suptitle("Student Performance — Deep-Dive EDA", fontsize=16, fontweight="bold")

axes[0, 0].hist(df["Number_of_Absences"], bins=20, color="#e67e22", edgecolor="white")
axes[0, 0].set_title("Absences Distribution"); axes[0, 0].set_xlabel("Number of Absences")

sns.violinplot(data=df, x="Mother_Education", y="Grade_1", palette=PALETTE, ax=axes[0, 1])
axes[0, 1].set_title("Mother's Education vs Grade 1")
axes[0, 1].set_xlabel("Mother Education Level (0–4)")

sns.boxplot(data=df, x="Internet_Access", y="Grade_1", palette=PALETTE, ax=axes[0, 2])
axes[0, 2].set_title("Internet Access vs Grade 1")

axes[1, 0].scatter(df["Grade_1"], df["Grade_2"], alpha=0.4, c="#2980b9")
axes[1, 0].set_title("Grade 1 vs Grade 2 Progression")
axes[1, 0].set_xlabel("Grade 1"); axes[1, 0].set_ylabel("Grade 2")

axes[1, 1].scatter(df["Weekend_Alcohol_Consumption"], df["Number_of_Absences"],
                   alpha=0.4, c="#c0392b")
axes[1, 1].set_title("Weekend Alcohol vs Absences")
axes[1, 1].set_xlabel("Weekend Alcohol (1–5)"); axes[1, 1].set_ylabel("Number of Absences")

sup_drop = df.groupby("School_Support")["Dropped_Out"].mean() * 100
axes[1, 2].bar(sup_drop.index, sup_drop.values, color=["#27ae60", "#c0392b"], edgecolor="white")
axes[1, 2].set_title("School Support vs Dropout Rate (%)")
axes[1, 2].set_ylabel("Dropout %")

plt.tight_layout()
plt.savefig("eda_deepdive.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → Saved: eda_deepdive.png")

/tmp/ipykernel_3841/3720711684.py:40: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_plot, x="Study_Time_Label", y="Grade_1",
/tmp/ipykernel_3841/3720711684.py:40: UserWarning: The palette list has more values (5) than needed (4), which may not be intended.
  sns.boxplot(data=df_plot, x="Study_Time_Label", y="Grade_1",
/tmp/ipykernel_3841/3720711684.py:47: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="Number_of_Failures", y="Grade_1", palette=PALETTE, ax=ax5)
/tmp/ipykernel_3841/3720711684.py:47: UserWarning: The palette list has more values (5) than needed (4), which may not be intended.
  sns.boxplot(data=df, x="Number_of_Failures", y="Grade_1", palette=PALETTE, ax=ax5

  → Saved: eda_overview.png


/tmp/ipykernel_3841/3720711684.py:71: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x="Mother_Education", y="Grade_1", palette=PALETTE, ax=axes[0, 1])
/tmp/ipykernel_3841/3720711684.py:75: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="Internet_Access", y="Grade_1", palette=PALETTE, ax=axes[0, 2])
/tmp/ipykernel_3841/3720711684.py:75: UserWarning: The palette list has more values (5) than needed (2), which may not be intended.
  sns.boxplot(data=df, x="Internet_Access", y="Grade_1", palette=PALETTE, ax=axes[0, 2])


  → Saved: eda_deepdive.png


# **Preprocessing**

In [5]:
df_pre = df.copy()

# 3-A  Remove Final_Grade to prevent data leakage
df_pre.drop(columns=["Final_Grade"], inplace=True)
print("[3-A] Removed 'Final_Grade' — prevents data leakage.")
print("      Rationale: Final grade is only known after dropout occurs.")
print("      Model uses Grade_1, Grade_2, and behavioural features instead.")

# 3-B  Encode binary Yes/No columns → 1/0
binary_cols = [
    "School_Support", "Family_Support", "Extra_Paid_Class",
    "Extra_Curricular_Activities", "Attended_Nursery",
    "Wants_Higher_Education", "Internet_Access", "In_Relationship"
]
for col in binary_cols:
    df_pre[col] = df_pre[col].map({"yes": 1, "no": 0})
print("\n[3-B] Binary encoding applied to yes/no columns.")

# 3-C  Encode nominal categorical columns with LabelEncoder
le = LabelEncoder()
nominal_cols = ["School", "Gender", "Address", "Family_Size",
                "Parental_Status", "Mother_Job", "Father_Job",
                "Reason_for_Choosing_School", "Guardian"]
encoders = {}
for col in nominal_cols:
    df_pre[col] = le.fit_transform(df_pre[col].astype(str))
    encoders[col] = le
print("[3-C] Label encoding applied to nominal categorical columns.")

# 3-D  Encode target variable
df_pre["Dropped_Out"] = df_pre["Dropped_Out"].astype(int)
print("[3-D] Target variable encoded: False → 0, True → 1.")

# 3-E  Feature Engineering
df_pre["Avg_Grade"]         = (df_pre["Grade_1"] + df_pre["Grade_2"]) / 2
df_pre["Grade_Improvement"] = df_pre["Grade_2"] - df_pre["Grade_1"]
df_pre["Total_Alcohol"]     = df_pre["Weekend_Alcohol_Consumption"] + df_pre["Weekday_Alcohol_Consumption"]
df_pre["Parent_Edu_Avg"]    = (df_pre["Mother_Education"] + df_pre["Father_Education"]) / 2
df_pre["Support_Score"]     = df_pre["School_Support"] + df_pre["Family_Support"]
df_pre["Risk_Score"]        = (df_pre["Number_of_Failures"] * 2 +
                                df_pre["Number_of_Absences"] / 5 +
                                df_pre["Total_Alcohol"] / 2)
print("\n[3-E] Feature engineering — 6 new derived features created:")
print("      • Avg_Grade         : Mean of Grade_1 and Grade_2")
print("      • Grade_Improvement : Grade_2 minus Grade_1 (trend)")
print("      • Total_Alcohol     : Weekend + Weekday consumption")
print("      • Parent_Edu_Avg    : Average parental education level")
print("      • Support_Score     : School + Family support combined")
print("      • Risk_Score        : Composite dropout risk indicator")

# 3-F  Outlier capping using IQR method
num_features_all = df_pre.select_dtypes(include=np.number).columns.drop("Dropped_Out")
outlier_report = {}
for col in num_features_all:
    Q1, Q3 = df_pre[col].quantile(0.25), df_pre[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((df_pre[col] < lower) | (df_pre[col] > upper)).sum()
    if n_out > 0:
        df_pre[col] = df_pre[col].clip(lower, upper)
        outlier_report[col] = n_out
print(f"\n[3-F] Outlier capping (IQR) applied to {len(outlier_report)} columns.")

# 3-G  Feature / Target split
X = df_pre.drop(columns=["Dropped_Out"])
y = df_pre["Dropped_Out"]

# 3-H  Feature Selection — top 20 by Mutual Information
selector = SelectKBest(mutual_info_classif, k=20)
selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()].tolist()
X = X[selected_features]
print(f"\n[3-H] Feature selection (Mutual Information) — top 20 selected:")
print("     ", selected_features)

# 3-I  Train/Test Split — 80/20 stratified
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"\n[3-I] Train/Test split: {X_train.shape[0]} train | {X_test.shape[0]} test")
print("      80/20 ratio chosen — standard split; large enough train set")
print("      for complex models while keeping 20% unseen for evaluation.")

# 3-J  Scaling with StandardScaler
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print("[3-J] StandardScaler applied — zero mean, unit variance.")


[3-A] Removed 'Final_Grade' — prevents data leakage.
      Rationale: Final grade is only known after dropout occurs.
      Model uses Grade_1, Grade_2, and behavioural features instead.

[3-B] Binary encoding applied to yes/no columns.
[3-C] Label encoding applied to nominal categorical columns.
[3-D] Target variable encoded: False → 0, True → 1.

[3-E] Feature engineering — 6 new derived features created:
      • Avg_Grade         : Mean of Grade_1 and Grade_2
      • Grade_Improvement : Grade_2 minus Grade_1 (trend)
      • Total_Alcohol     : Weekend + Weekday consumption
      • Parent_Edu_Avg    : Average parental education level
      • Support_Score     : School + Family support combined
      • Risk_Score        : Composite dropout risk indicator

[3-F] Outlier capping (IQR) applied to 22 columns.

[3-H] Feature selection (Mutual Information) — top 20 selected:
      ['School', 'Gender', 'Address', 'Father_Education', 'Mother_Job', 'Number_of_Failures', 'School_Support', 'Atte

# **Baseline Models**

In [6]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest":        RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting":    GradientBoostingClassifier(random_state=42),
    "SVM (RBF)":            SVC(kernel="rbf", probability=True, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
baseline_results = {}

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_sc, y_train, cv=cv, scoring="roc_auc")
    model.fit(X_train_sc, y_train)
    preds = model.predict(X_test_sc)
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, model.predict_proba(X_test_sc)[:, 1])
    baseline_results[name] = {"CV AUC": cv_scores.mean(), "Test Acc": acc, "Test AUC": auc}
    print(f"  {name:25s} | CV-AUC: {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f} | "
          f"Test Acc: {acc:.4f} | Test AUC: {auc:.4f}")

  Logistic Regression       | CV-AUC: 0.9672 ± 0.0238 | Test Acc: 0.9000 | Test AUC: 0.9500
  Random Forest             | CV-AUC: 0.9729 ± 0.0161 | Test Acc: 0.9231 | Test AUC: 0.9514
  Gradient Boosting         | CV-AUC: 0.9586 ± 0.0241 | Test Acc: 0.8923 | Test AUC: 0.9295
  SVM (RBF)                 | CV-AUC: 0.9550 ± 0.0310 | Test Acc: 0.9462 | Test AUC: 0.9550


# **Hyperparameter Fine Tuning**

In [7]:
# 6-A  Random Forest
print("\n[6-A] Random Forest — GridSearchCV")
rf_params = {
    "n_estimators":      [100, 200, 300],
    "max_depth":         [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "max_features":      ["sqrt", "log2"],
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42),
                       rf_params, cv=cv, scoring="roc_auc", n_jobs=-1, verbose=0)
rf_grid.fit(X_train_sc, y_train)
rf_best = rf_grid.best_estimator_
print(f"  Best params : {rf_grid.best_params_}")
print(f"  Best CV-AUC : {rf_grid.best_score_:.4f}")

# 6-B  Gradient Boosting
print("\n[6-B] Gradient Boosting — GridSearchCV")
gb_params = {
    "n_estimators":  [100, 200],
    "max_depth":     [3, 5, 7],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample":     [0.8, 1.0],
}
gb_grid = GridSearchCV(GradientBoostingClassifier(random_state=42),
                       gb_params, cv=cv, scoring="roc_auc", n_jobs=-1, verbose=0)
gb_grid.fit(X_train_sc, y_train)
gb_best = gb_grid.best_estimator_
print(f"  Best params : {gb_grid.best_params_}")
print(f"  Best CV-AUC : {gb_grid.best_score_:.4f}")

# 6-C  Logistic Regression
print("\n[6-C] Logistic Regression — GridSearchCV")
lr_params = {
    "C":      [0.01, 0.1, 1, 10, 100],
    "solver": ["lbfgs", "liblinear"],
    "penalty":["l2"],
}
lr_grid = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42),
                       lr_params, cv=cv, scoring="roc_auc", n_jobs=-1, verbose=0)
lr_grid.fit(X_train_sc, y_train)
lr_best = lr_grid.best_estimator_
print(f"  Best params : {lr_grid.best_params_}")
print(f"  Best CV-AUC : {lr_grid.best_score_:.4f}")


[6-A] Random Forest — GridSearchCV
  Best params : {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 300}
  Best CV-AUC : 0.9751

[6-B] Gradient Boosting — GridSearchCV
  Best params : {'learning_rate': 0.05, 'max_depth': 7, 'n_estimators': 200, 'subsample': 0.8}
  Best CV-AUC : 0.9722

[6-C] Logistic Regression — GridSearchCV
  Best params : {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}
  Best CV-AUC : 0.9680


# **Final Evaluation**

In [8]:
tuned_models = {
    "Logistic Regression (Tuned)": lr_best,
    "Random Forest (Tuned)":       rf_best,
    "Gradient Boosting (Tuned)":   gb_best,
}

final_results = {}
for name, model in tuned_models.items():
    preds = model.predict(X_test_sc)
    proba = model.predict_proba(X_test_sc)[:, 1]
    acc   = accuracy_score(y_test, preds)
    auc   = roc_auc_score(y_test, proba)
    final_results[name] = {"Test Acc": acc, "Test AUC": auc, "proba": proba}
    print(f"\n  {'─'*45}")
    print(f"  {name}")
    print(f"  {'─'*45}")
    print(f"  Accuracy : {acc:.4f}   ROC-AUC : {auc:.4f}")
    print(classification_report(y_test, preds, target_names=["Not Dropped", "Dropped Out"]))

best_name  = max(final_results, key=lambda k: final_results[k]["Test AUC"])
best_model = tuned_models[best_name]
print(f"\n  ★ Best Model : {best_name}")
print(f"    Test AUC   : {final_results[best_name]['Test AUC']:.4f}")
print(f"    Test Acc   : {final_results[best_name]['Test Acc']:.4f}")


  ─────────────────────────────────────────────
  Logistic Regression (Tuned)
  ─────────────────────────────────────────────
  Accuracy : 0.9308   ROC-AUC : 0.9486
              precision    recall  f1-score   support

 Not Dropped       0.95      0.96      0.96       110
 Dropped Out       0.79      0.75      0.77        20

    accuracy                           0.93       130
   macro avg       0.87      0.86      0.86       130
weighted avg       0.93      0.93      0.93       130


  ─────────────────────────────────────────────
  Random Forest (Tuned)
  ─────────────────────────────────────────────
  Accuracy : 0.9077   ROC-AUC : 0.9559
              precision    recall  f1-score   support

 Not Dropped       0.96      0.93      0.94       110
 Dropped Out       0.67      0.80      0.73        20

    accuracy                           0.91       130
   macro avg       0.81      0.86      0.84       130
weighted avg       0.92      0.91      0.91       130


  ─────────────────

# **Evaluation Plots**

In [9]:
fig3, axes3 = plt.subplots(1, 3, figsize=(18, 5))
fig3.suptitle("Model Evaluation Results (Post SMOTE + Fine-Tuning)", fontsize=14, fontweight="bold")

# Confusion Matrix — best model
cm = confusion_matrix(y_test, best_model.predict(X_test_sc))
disp = ConfusionMatrixDisplay(cm, display_labels=["Not Dropped", "Dropped Out"])
disp.plot(ax=axes3[0], cmap="Blues", colorbar=False)
axes3[0].set_title(f"Confusion Matrix\n{best_name}", fontweight="bold")

# ROC Curves — all tuned models
for name, model in tuned_models.items():
    proba = model.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes3[1].plot(fpr, tpr, label=f"{name} (AUC={final_results[name]['Test AUC']:.3f})")
axes3[1].plot([0, 1], [0, 1], "k--", label="Random Classifier")
axes3[1].set_title("ROC Curves — Tuned Models", fontweight="bold")
axes3[1].set_xlabel("False Positive Rate")
axes3[1].set_ylabel("True Positive Rate")
axes3[1].legend(fontsize=7)

# Feature Importances — Random Forest
fi = pd.Series(rf_best.feature_importances_, index=selected_features).sort_values(ascending=True)
fi.plot(kind="barh", ax=axes3[2], color="#3498db", edgecolor="white")
axes3[2].set_title("Feature Importances\n(Random Forest Tuned)", fontweight="bold")
axes3[2].set_xlabel("Importance Score")

plt.tight_layout()
plt.savefig("model_evaluation.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → Saved: model_evaluation.png")

# Baseline vs Tuned model comparison
fig4, ax4 = plt.subplots(figsize=(11, 6))
all_names = list(baseline_results.keys()) + list(final_results.keys())
all_aucs  = ([baseline_results[n]["Test AUC"] for n in baseline_results] +
             [final_results[n]["Test AUC"]     for n in final_results])
colors = ["#95a5a6"] * 4 + ["#2ecc71", "#3498db", "#e74c3c"]
bars = ax4.barh(all_names, all_aucs, color=colors, edgecolor="white")
ax4.set_xlim(0.5, 1.0)
ax4.set_xlabel("Test ROC-AUC")
ax4.set_title("Baseline vs Fine-Tuned — Model Comparison\n(Grey = Baseline | Coloured = Tuned)",
              fontweight="bold")
for bar, val in zip(bars, all_aucs):
    ax4.text(val + 0.003, bar.get_y() + bar.get_height() / 2,
             f"{val:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → Saved: model_comparison.png")

  → Saved: model_evaluation.png
  → Saved: model_comparison.png


In [10]:
joblib.dump(best_model, "best_model.pkl")
joblib.dump(scaler,     "scaler.pkl")
print("\n[9] Saved: best_model.pkl, scaler.pkl")


[9] Saved: best_model.pkl, scaler.pkl


In [11]:
print("\n" + "=" * 60)
print("  RESEARCH PIPELINE SUMMARY")
print("=" * 60)
print(f"  Dataset          : {df.shape[0]} students, {df.shape[1]} original features")
print(f"  Target           : Dropped_Out ({df['Dropped_Out'].mean()*100:.1f}% positive)")
print(f"  Final_Grade      : EXCLUDED (data leakage prevention)")
print(f"  Features used    : {len(selected_features)} (after feature selection)")
print(f"  Train / Test     : {X_train.shape[0]} / {X_test.shape[0]} (80/20, stratified)")
print(f"  Models compared  : 4 baseline + 3 fine-tuned")
print(f"  Best Model       : {best_name}")
print(f"  Best Test AUC    : {final_results[best_name]['Test AUC']:.4f}")
print(f"  Best Test Acc    : {final_results[best_name]['Test Acc']:.4f}")
print("=" * 60)
print("\nAll outputs saved. Pipeline complete ✔")


  RESEARCH PIPELINE SUMMARY
  Dataset          : 649 students, 34 original features
  Target           : Dropped_Out (15.4% positive)
  Final_Grade      : EXCLUDED (data leakage prevention)
  Features used    : 20 (after feature selection)
  Train / Test     : 519 / 130 (80/20, stratified)
  Models compared  : 4 baseline + 3 fine-tuned
  Best Model       : Random Forest (Tuned)
  Best Test AUC    : 0.9559
  Best Test Acc    : 0.9077

All outputs saved. Pipeline complete ✔
